## 02 - raw -> bronze

Finds raw pages that are in `ctl_api_call_log` but not in bronze yet, unpacks the Bundles and appends one row per resource to `bronze_<resource>` (Delta).

The resource stays as json text in `resource_json`, so there is no schema inference on the (messy) test server data - silver does the typed parsing. XML pages are converted to the same json shape first (`fhir_lakehouse/fhir_xml.py`), everything after this step doesn't care about the format.

Every row carries `extraction_timestamp`, `api_url_or_params`, the raw file path, the ingest run id and `bronze_loaded_at`.

In [ ]:
resource_type = "Patient"
run_id = ""
config_path = "/lakehouse/default/Files/config/pipeline_config.json"
code_path = "/lakehouse/default/Files/code"
lakehouse_root = ""    # blank: use the default lakehouse attached to the notebook

In [ ]:
import json
import sys

if code_path and code_path not in sys.path:
    sys.path.insert(0, code_path)

from pyspark.sql import functions as F

from fhir_lakehouse.bronze import raw_to_bronze
from fhir_lakehouse.config import apply_lakehouse_root, get_resource, load_config, table_name
from fhir_lakehouse.control import StepLog, new_run_id
from fhir_lakehouse.storage import Lakehouse, get_spark

cfg = apply_lakehouse_root(load_config(config_path), lakehouse_root)
resource = get_resource(cfg, resource_type)
run_id = run_id or new_run_id()

spark = get_spark()
spark.conf.set("spark.sql.session.timeZone", cfg["spark"]["timezone"])
lh = Lakehouse(spark, cfg)

In [ ]:
step = StepLog(lh, cfg, run_id, "raw_to_bronze", resource["name"])
try:
    stats = raw_to_bronze(lh, cfg, resource["name"], run_id)
    step.finish("SUCCEEDED", rows_read=stats["rows_read"], rows_written=stats["rows_written"], details=stats)
except Exception as e:
    step.finish("FAILED", error=repr(e))
    raise

In [ ]:
(lh.read(table_name(cfg, "bronze", resource["name"]))
   .groupBy("load_date", "ingest_run_id")
   .agg(F.count("*").alias("rows"), F.countDistinct("resource_id").alias("ids"))
   .orderBy("load_date")
   .show(truncate=False))

In [ ]:
result = json.dumps(stats, default=str)
print(result)
try:
    notebookutils.notebook.exit(result)  # shows up as the activity output in the pipeline
except NameError:
    pass  # not running in Fabric